# Case 02: how many model calls does a retry cost?

A result notebook that needs no credentials. It repeats one experiment many times against a fake, flaky backend to check the retry contract in `pilot_kit.retry`: **one logical call makes at most `DEFAULT_ATTEMPTS` model calls**, and permanent errors make exactly one.

All code cells are tagged `offline`, so CI re-runs them on every change.

In [1]:
import asyncio
import logging
import pathlib
import random
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make src/ importable from notebooks/

from pilot_kit.retry import DEFAULT_ATTEMPTS, with_retries

# One warning per retry would flood the output.
logging.getLogger("pilot_kit.retry").setLevel(logging.ERROR)
print(f"DEFAULT_ATTEMPTS = {DEFAULT_ATTEMPTS}")

DEFAULT_ATTEMPTS = 3


## 1. One logical call

The fake backend resets the connection with probability `fail_p`, and counts every call it receives. `delay=0` skips the waits.

In [2]:
class Backend:
    def __init__(self, fail_p, rng, error=ConnectionResetError):
        self.fail_p, self.rng, self.error, self.calls = fail_p, rng, error, 0

    async def __call__(self):
        self.calls += 1
        if self.rng.random() < self.fail_p:
            raise self.error("backend failed")
        return "ok"


async def logical_call(fail_p, rng, error=ConnectionResetError):
    backend = Backend(fail_p, rng, error)
    try:
        await with_retries(backend, delay=0)
    except (ConnectionError, ValueError):
        return backend.calls, False
    return backend.calls, True


print(await logical_call(0.5, random.Random(1)))

(2, True)


## 2. Repeated: transient failures

For each failure rate, **200 logical calls** with a seeded random generator. The table shows how often the call still succeeds and how many model calls it cost.

In [3]:
RUNS = 200
rows = []
for fail_p in (0.0, 0.2, 0.5, 0.8):
    rng = random.Random(7)
    results = await asyncio.gather(*[logical_call(fail_p, rng) for _ in range(RUNS)])
    calls = [c for c, _ in results]
    ok = sum(1 for _, success in results if success)
    assert max(calls) <= DEFAULT_ATTEMPTS
    rows.append((fail_p, ok, sum(calls) / RUNS, max(calls)))

print("| Failure rate | Succeeded | Mean model calls | Max model calls |")
print("|---|---|---|---|")
for fail_p, ok, mean_calls, max_calls in rows:
    print(f"| {fail_p:.1f} | {ok}/{RUNS} | {mean_calls:.2f} | {max_calls} |")

| Failure rate | Succeeded | Mean model calls | Max model calls |
|---|---|---|---|
| 0.0 | 200/200 | 1.00 | 1 |
| 0.2 | 197/200 | 1.28 | 3 |
| 0.5 | 174/200 | 1.79 | 3 |
| 0.8 | 96/200 | 2.42 | 3 |


## 3. Repeated: permanent failures

A `ValueError` stands in for an error that will not fix itself, such as a rejected key. **200 logical calls** that always fail.

In [4]:
results = await asyncio.gather(
    *[logical_call(1.0, random.Random(seed), ValueError) for seed in range(RUNS)]
)
calls = {c for c, _ in results}
failed = sum(1 for _, success in results if not success)
print(f"model calls per logical call: {sorted(calls)}; failed: {failed}/{RUNS}")
assert calls == {1}

model calls per logical call: [1]; failed: 200/200


## Result

With 20% of calls failing, almost every logical call still succeeds after a retry. Even when 80% fail, no logical call ever makes more than `DEFAULT_ATTEMPTS` model calls, so the cost of an outage is bounded. A permanent error makes exactly one call and is never repeated. The numbers come from a seeded fake backend, so they are the same on every run.